# Buổi 2 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5).

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Chạy code có sẵn, thấy hai triệu chứng

In [ ]:
%run xac_suat.py

## Bước 2 — Tương quan và kiểm định hoán vị trên dữ liệu thật (mục 4.4, 4.5)

In [ ]:
import numpy as np
import pandas as pd
from xac_suat import doc_luot_thue

import tv  # thư viện trợ giúp của khoá, biết chỗ để dữ liệu

h = doc_luot_thue()
print(np.corrcoef(h["temp"], h["cnt"])[0, 1])             # nhiệt độ × lượt thuê, mọi giờ

In [ ]:
def hoan_vi(y, nhom, so_lan=9999, seed=2026):             # hàm của mục 4.5
    y, nhom = np.asarray(y, dtype=float), np.asarray(nhom, dtype=bool)
    that = y[nhom].mean() - y[~nhom].mean()
    rng = np.random.default_rng(seed)
    k = 0
    for _ in range(so_lan):
        xao = rng.permutation(nhom)
        k += abs(y[xao].mean() - y[~xao].mean()) >= abs(that)
    return that, (k + 1) / (so_lan + 1)


d = pd.read_csv(tv.THU_MUC_DU_LIEU / "uci-bike-sharing" / "day.csv")
n12 = d[d["yr"] == 1]                                     # yr = 1 là năm 2012
print(hoan_vi(n12["cnt"], n12["workingday"] == 1))        # ngày làm việc − ngày nghỉ

## Bước 3 — Bootstrap tự viết so với scipy (bảng "Thư viện" mục 4.6)

In [ ]:
from scipy import stats
from xac_suat import ar1, khoang_tin_cay_trung_binh

x = ar1(200, 0.7, np.random.default_rng(7))
print(khoang_tin_cay_trung_binh(x, so_lan=9999, do_dai_khoi=1, seed=7))    # tự viết, i.i.d.
kq = stats.bootstrap((x,), np.mean, n_resamples=9999, method="percentile", rng=np.random.default_rng(7))
print(kq.confidence_interval)                                                # scipy, i.i.d.
print(khoang_tin_cay_trung_binh(x, so_lan=9999, do_dai_khoi=6, seed=7))    # tự viết, khối 6

## Bước 4 — Đo tỷ lệ phủ thật

Sửa `khoang_du_bao` trong `xac_suat.py` (tài liệu mục 5, bước 4) rồi chạy lại ô này.

In [ ]:
from xac_suat import khoang_theo_gio, ty_le_phu

nam_2011, nam_2012 = h[h["yr"] == 0], h[h["yr"] == 1]
k = khoang_theo_gio(nam_2011)                             # một khoảng cho mỗi giờ 0–23 (cột hr)
for ten, moi in [("2011 → 2011", nam_2011), ("2011 → 2012", nam_2012)]:
    m = moi.merge(k, on="hr")                             # gắn khoảng của đúng giờ đó vào từng dòng
    print(ten, ty_le_phu(m["cnt"], m["lo"], m["hi"]))

## Bước 5 — Block bootstrap

Sửa `khoang_tin_cay_trung_binh` (tài liệu mục 5, bước 5) rồi chạy ô này. Chấm: `python lab.py check` trong terminal.

In [ ]:
from xac_suat import ty_le_phu_khoang_tin_cay

for khoi in (1, 3, 6, 10, 20, 40):   # rho = 0,7, n = 200, 300 chuỗi, seed 2026 (mặc định của hàm)
    print(khoi, ty_le_phu_khoang_tin_cay(do_dai_khoi=khoi))